# Transfer-Pricing Bulk Research with Foundry + Bing Grounding (GA)

This notebook provisions and exercises a **single Foundry PromptAgent** —
`TransferPricingComparablesAgent` — for the **bulk-company research** use case:

> Generate structured business descriptions for hundreds-to-thousands of
> comparable companies as part of a transfer-pricing analysis.

The companion notebook `regulatory-monitoring.ipynb` covers the
regulatory-monitoring use case with a separate agent.

## What's in this notebook

| Section | What it does |
|---|---|
| Provisioning | Optional one-time setup of the Foundry account / project / model / Bing connection. |
| Configure | Loads endpoint + connection IDs from a `.env` file. |
| Provision agent | Creates the GA `TransferPricingComparablesAgent` PromptAgent server-side. |
| Single run demo | Sanity-check the agent on one prompt. |
| Bulk runner | Scales the agent to hundreds/thousands of companies with concurrency, retries, and a resumable JSONL checkpoint. |
| Citation guardrail (3 layers) | Groundedness, URL reachability, destination safety. |
| Evaluation | Batch eval harness with built-in groundedness scoring. |

Only GA features are used:

- `agent-framework-foundry` (released) and its `FoundryAgent` / `FoundryChatClient`.
- `azure-ai-projects` >= 2.0 with `BingGroundingTool` (GA).
- Foundry project/connections control-plane API version `2025-06-01` (GA).
- Model deployment `gpt-5.1` (GA), model version `2025-11-13`.
- Bing resource API `2020-06-10` (GA).

## Prerequisites

1. An **Azure AI Foundry** project (endpoint from the Foundry portal Overview page).
2. A model deployment (e.g. `gpt-5.1`).
3. A **Grounding with Bing Search** resource connected to your Foundry project.
4. Azure CLI logged in: `az login`.

## Install


In [ ]:
# Install the runtime dependencies for this notebook.
#   - agent-framework-foundry : GA Microsoft Agent Framework + Foundry connectors
#                               (provides FoundryAgent, FoundryChatClient, etc.)
#   - azure-ai-projects>=2.0  : GA Foundry SDK (PromptAgent, BingGroundingTool)
#   - azure-identity          : DefaultAzureCredential / AzureCliCredential for auth
#   - httpx                   : async HTTP client used by the citation guardrails
#   - python-dotenv           : load endpoint/connection IDs from a local .env file
#   - azure-ai-contentsafety  : groundedness + destination safety checks
#   - azure-ai-evaluation     : batch evaluation harness for citation quality
%pip install --upgrade "agent-framework-foundry" "azure-ai-projects>=2.0.0" azure-identity httpx python-dotenv azure-ai-contentsafety azure-ai-evaluation


## (Optional) Provision Azure resources if they don't exist

This section deploys the Foundry + Grounding-with-Bing-Search infrastructure
declared in [`infra/main.bicep`](./infra/main.bicep), which composes two
reusable modules:

- `modules/foundry/foundry.bicep`         — AI Services account + project + model deployment
- `modules/bing-grounding/bing-grounding.bicep` — Bing.Grounding account + project connection

All notebooks share the same infrastructure, so this
template is the single source of truth — re-running it is a no-op.

Requirements:

- Azure CLI logged in: `az login`
- `.env` contains `AZURE_SUBSCRIPTION_ID` (plus any overrides for resource
  group, region, account names, etc. — see `.env.example`)

Skip this section if your Foundry project, model deployment, and Bing
connection already exist — the **Configure** cell below will read them from
`.env`.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Provisioning configuration (this whole section is OPTIONAL)
#
# Values are read from environment variables (typically a .env file at the
# repo root — see .env.example). Edit .env, not this cell.
#
# Required ONLY if you actually deploy infra/main.bicep:
#   AZURE_SUBSCRIPTION_ID
#
# All others have sensible defaults you can override in .env:
#   RESOURCE_GROUP, LOCATION,
#   FOUNDRY_ACCOUNT, FOUNDRY_PROJECT,
#   MODEL_NAME, MODEL_VERSION, MODEL_DEPLOYMENT, MODEL_SKU, MODEL_CAPACITY,
#   BING_RESOURCE, BING_SKU, BING_CONNECTION_NAME
#
# If your Foundry project / model deployment / Bing connection already exist,
# you can skip this entire section and jump straight to the Configure cell
# below — Configure loads .env independently.
# ─────────────────────────────────────────────────────────────────────────────
import os

try:
    from dotenv import find_dotenv, load_dotenv
    _dotenv_path = find_dotenv(usecwd=True)
    if _dotenv_path:
        load_dotenv(_dotenv_path, override=False)
        print(f"dotenv: loaded from {_dotenv_path}")
    else:
        print("Warning: no .env file found via find_dotenv(); relying on process environment.")
except ImportError:
    print("python-dotenv not installed; relying on process environment.")


def _env(name: str, default: str) -> str:
    """Read an env var, falling back to `default` when unset or blank."""
    val = (os.environ.get(name) or "").strip()
    return val or default


# Subscription is only needed if you run the deploy cell below.
# We don't raise here — let users skip provisioning entirely.
SUBSCRIPTION_ID = (os.environ.get("AZURE_SUBSCRIPTION_ID") or "").strip()
if not SUBSCRIPTION_ID:
    print(
        "\nNote: AZURE_SUBSCRIPTION_ID is not set. That's fine if you're "
        "skipping provisioning. If you intend to run the deploy cell below, "
        "set it in .env first."
    )

# Resource group + region.
RESOURCE_GROUP = _env("RESOURCE_GROUP", "rg-foundry-bing-research")
LOCATION = _env("LOCATION", "eastus")

# Foundry (AI Services) account + project names.
FOUNDRY_ACCOUNT = _env("FOUNDRY_ACCOUNT", "foundry-bing-research-acct")
FOUNDRY_PROJECT = _env("FOUNDRY_PROJECT", "bing-research-proj")

# Model deployment settings.
MODEL_NAME = _env("MODEL_NAME", "gpt-5.1")
MODEL_VERSION = _env("MODEL_VERSION", "2025-11-13")
MODEL_DEPLOYMENT = _env("MODEL_DEPLOYMENT", "gpt-5.1")
MODEL_SKU = _env("MODEL_SKU", "GlobalStandard")
MODEL_CAPACITY = int(_env("MODEL_CAPACITY", "50"))

# Grounding-with-Bing-Search settings.
BING_RESOURCE = _env("BING_RESOURCE", "bing-grounding-research")
BING_SKU = _env("BING_SKU", "G1")
BING_CONNECTION_NAME = _env("BING_CONNECTION_NAME", "bing-grounding-conn")

# Foundry control-plane API version (code-level constant; bumping is a code change).
FOUNDRY_API_VERSION = "2025-06-01"

print(f"\nSUBSCRIPTION_ID      = {SUBSCRIPTION_ID or '(unset — skipping provisioning is OK)'}")
print(f"RESOURCE_GROUP       = {RESOURCE_GROUP}  (LOCATION={LOCATION})")
print(f"FOUNDRY_ACCOUNT      = {FOUNDRY_ACCOUNT}  (project={FOUNDRY_PROJECT})")
print(f"MODEL                = {MODEL_NAME} v{MODEL_VERSION}  deployment={MODEL_DEPLOYMENT}  sku={MODEL_SKU} capacity={MODEL_CAPACITY}")
print(f"BING_RESOURCE        = {BING_RESOURCE}  sku={BING_SKU}  connection={BING_CONNECTION_NAME}")


In [ ]:
import json
import shlex
import shutil
import subprocess


# Resolve the actual az executable once. On Windows, `az` ships as `az.cmd`
# (a batch wrapper), and subprocess.run([...], shell=False) does NOT search
# PATHEXT — passing the bare string "az" raises FileNotFoundError (WinError 2).
# shutil.which respects PATHEXT and returns the full path to az.cmd / az.
_AZ_BIN = shutil.which("az")
if not _AZ_BIN:
    raise RuntimeError(
        "Could not find the Azure CLI on PATH. Install it from "
        "https://learn.microsoft.com/cli/azure/install-azure-cli and "
        "restart the kernel."
    )


def az(*args: str, check: bool = True, parse_json: bool = True):
    """Run an `az` CLI command and return its parsed JSON output.

    Why a wrapper instead of calling subprocess directly?
      * Adds `--only-show-errors` so progress logs don't pollute notebook output.
      * Centralizes JSON parsing and error handling.
      * `check=False` returns None on non-zero exit (used to detect "resource missing").

    Args:
        *args: Tokens passed to `az` (e.g. az("group", "show", "-n", "rg")).
        check: When True (default), raise on non-zero exit. When False, return None.
        parse_json: When True, parse stdout as JSON and return the dict/list.
                    Set False for commands whose output isn't JSON (e.g. `account set`).
    """
    # Always suppress non-error chatter for cleaner notebook output.
    cmd = [_AZ_BIN, *args, "--only-show-errors"]
    if parse_json:
        # Force JSON output mode so json.loads() works regardless of user defaults.
        cmd += ["-o", "json"]
    # Echo as `az ...` (not the resolved path) for readability.
    print("$ " + " ".join(shlex.quote(c) for c in ["az", *args, "--only-show-errors"] + (["-o", "json"] if parse_json else [])))
    # shell=False because we resolved the full executable path above.
    res = subprocess.run(cmd, capture_output=True, text=True, shell=False)
    if res.returncode != 0:
        if check:
            # Surface stderr so the caller can see what az actually complained about.
            raise RuntimeError(f"az failed ({res.returncode}): {res.stderr.strip()}")
        # check=False — let the caller treat None as "resource doesn't exist".
        return None
    if not parse_json:
        return res.stdout.strip()
    # Empty stdout (e.g. a 204) shouldn't crash json.loads().
    return json.loads(res.stdout) if res.stdout.strip() else None




In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Deploy ./infra/main.bicep into RESOURCE_GROUP, then surface its outputs as
# environment variables so the Configure cell below picks them up.
# Idempotent — Bicep diffs against deployed state and only changes what differs.
# ─────────────────────────────────────────────────────────────────────────────
if not SUBSCRIPTION_ID:
    raise RuntimeError(
        "AZURE_SUBSCRIPTION_ID is required to deploy. Set it in .env and "
        "re-run the provisioning configuration cell above. (If you don't "
        "want to provision, skip this cell — the Configure cell below "
        "reads from .env independently.)"
    )

# Pin the active subscription so every `az` call targets the right one.
az("account", "set", "--subscription", SUBSCRIPTION_ID, parse_json=False)
# The `cognitiveservices` extension is required for Foundry-specific commands.
az("extension", "add", "--name", "cognitiveservices", "--upgrade", parse_json=False, check=False)

from pathlib import Path

# Ensure the resource group exists. (RG creation is outside the Bicep template's
# resource-group scope, so it's handled here.)
if az("group", "show", "-n", RESOURCE_GROUP, check=False) is None:
    az("group", "create", "-n", RESOURCE_GROUP, "-l", LOCATION)
    print(f"Created resource group: {RESOURCE_GROUP}")
else:
    print(f"Resource group already exists: {RESOURCE_GROUP}")

# Locate the Bicep template — repo layout: <repo>/infra/main.bicep.
# find_dotenv-style search up the tree handles both "notebook at repo root"
# and "notebook in labs/<lab>/" layouts.
def _find_bicep() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        candidate = parent / "infra" / "main.bicep"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate infra/main.bicep. Run this notebook from the repo "
        "(or any subdirectory) where infra/main.bicep lives."
    )

bicep_path = _find_bicep()
print(f"\nDeploying {bicep_path} (idempotent — safe to re-run)...")

deployment = az(
    "deployment", "group", "create",
    "-g", RESOURCE_GROUP,
    "-f", str(bicep_path),
    "--parameters",
        f"location={LOCATION}",
        f"foundryAccountName={FOUNDRY_ACCOUNT}",
        f"foundryProjectName={FOUNDRY_PROJECT}",
        f"modelName={MODEL_NAME}",
        f"modelVersion={MODEL_VERSION}",
        f"modelDeploymentName={MODEL_DEPLOYMENT}",
        f"modelSku={MODEL_SKU}",
        f"modelCapacity={MODEL_CAPACITY}",
        f"bingResourceName={BING_RESOURCE}",
        f"bingSku={BING_SKU}",
        f"bingConnectionName={BING_CONNECTION_NAME}",
)

outputs = deployment["properties"]["outputs"]
PROJECT_ENDPOINT_RESOLVED = outputs["projectEndpoint"]["value"]
BING_CONNECTION_ID_RESOLVED = outputs["bingConnectionId"]["value"]
PROJECT_ID = outputs["projectId"]["value"]
BING_RESOURCE_ID = outputs["bingResourceId"]["value"]

# Surface the resolved values as env vars so the Configure cell picks them up
# (matches the env-var names the Configure cell looks for).
import os
os.environ["FOUNDRY_PROJECT_ENDPOINT"] = PROJECT_ENDPOINT_RESOLVED
os.environ["FOUNDRY_MODEL_NAME"] = MODEL_DEPLOYMENT
os.environ["BING_CONNECTION_ID"] = BING_CONNECTION_ID_RESOLVED

print("\n[OK] Deployment complete.")
print(f"  Project endpoint   : {PROJECT_ENDPOINT_RESOLVED}")
print(f"  Bing connection id : {BING_CONNECTION_ID_RESOLVED}")
print(f"  Model deployment   : {MODEL_DEPLOYMENT}")
print("\nEnvironment configured. Run the Configure cell below.")


## Configure

Set these via environment variables or edit inline. The variable names match the GA `azure-ai-projects` SDK conventions.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Configure
# Loads required settings from environment variables (typically via a .env file).
# Fails fast with actionable messages if anything is missing or malformed.
# ─────────────────────────────────────────────────────────────────────────────
import os
from urllib.parse import urlsplit, urlunsplit

try:
    from dotenv import load_dotenv, find_dotenv
except ImportError as e:
    raise ImportError(
        "python-dotenv is required; run the install cell at the top of the notebook."
    ) from e

_dotenv_path = find_dotenv(usecwd=True)
if _dotenv_path:
    load_dotenv(_dotenv_path, override=True)
    print(f"dotenv: loaded from {_dotenv_path}")
else:
    print("Warning: no .env file found via find_dotenv(); relying on process environment.")


def _first_env(*names: str, default: str | None = None) -> str | None:
    """Return the first non-empty value among the given env var names."""
    for n in names:
        v = os.environ.get(n)
        if v:
            return v
    return default


def _require(name: str, value: str | None, *, validator=None, hint: str = "") -> str:
    """Validate a required config value; raise with a clear message if missing/invalid."""
    if not value or value.startswith("<") and value.endswith(">"):
        raise ValueError(f"{name} is not set. {hint}".strip())
    if validator is not None:
        ok, msg = validator(value)
        if not ok:
            raise ValueError(f"{name} is invalid ({msg}): {value!r}. {hint}".strip())
    return value


# Foundry project endpoint, e.g.
#   https://<account>.services.ai.azure.com/api/projects/<project>
PROJECT_ENDPOINT = _require(
    "PROJECT_ENDPOINT",
    _first_env("FOUNDRY_PROJECT_ENDPOINT", "AZURE_AI_PROJECT_ENDPOINT"),
    validator=lambda v: (v.startswith("https://"), "must be an https:// URL"),
    hint="Set FOUNDRY_PROJECT_ENDPOINT (or AZURE_AI_PROJECT_ENDPOINT) in your .env file.",
)

# Deployment name of the GA model in your Foundry project.
MODEL_DEPLOYMENT_NAME = _require(
    "MODEL_DEPLOYMENT_NAME",
    _first_env("FOUNDRY_MODEL_NAME", "AZURE_AI_MODEL_DEPLOYMENT_NAME", default="gpt-5.1"),
    hint="Set FOUNDRY_MODEL_NAME (or AZURE_AI_MODEL_DEPLOYMENT_NAME) in your .env file.",
)

# Full ARM connection id of the Grounding-with-Bing-Search connection in your project.
BING_CONNECTION_ID = _require(
    "BING_CONNECTION_ID",
    _first_env("BING_PROJECT_CONNECTION_ID", "BING_CONNECTION_ID"),
    validator=lambda v: (v.startswith("/subscriptions/"), "must be a full ARM resource id"),
    hint="Set BING_PROJECT_CONNECTION_ID (or BING_CONNECTION_ID) in your .env file.",
)

# Logical name for the PromptAgent we'll create in the Foundry project.
# Overridable via env var so multiple notebooks can target distinct agents in the same project.
AGENT_NAME_DEFAULT = "TransferPricingComparablesAgent"
AGENT_NAME = os.environ.get("TP_AGENT_NAME", AGENT_NAME_DEFAULT)

# Azure AI Content Safety endpoint (used by the citation guardrail pipeline).
# AI Services accounts expose Content Safety on the same host as the Foundry project,
# so we derive scheme+host from PROJECT_ENDPOINT if a dedicated env var isn't set.
_parts = urlsplit(PROJECT_ENDPOINT)
CONTENT_SAFETY_ENDPOINT = _require(
    "CONTENT_SAFETY_ENDPOINT",
    os.environ.get("CONTENT_SAFETY_ENDPOINT") or urlunsplit((_parts.scheme, _parts.netloc, "", "", "")),
    validator=lambda v: (v.startswith("https://"), "must be an https:// URL"),
)


def _redact_arm(arm_id: str) -> str:
    """Mask the subscription guid when echoing ARM ids."""
    parts = arm_id.split("/")
    if len(parts) > 3 and parts[1] == "subscriptions":
        parts[2] = "<sub>"
    return "/".join(parts)


print(f"PROJECT_ENDPOINT       = {PROJECT_ENDPOINT}")
print(f"MODEL_DEPLOYMENT_NAME  = {MODEL_DEPLOYMENT_NAME}")
print(f"BING_CONNECTION_ID     = {_redact_arm(BING_CONNECTION_ID)}")
print(f"CONTENT_SAFETY_ENDPOINT= {CONTENT_SAFETY_ENDPOINT}")
print(f"AGENT_NAME             = {AGENT_NAME}")


## Provision the transfer-pricing comparables PromptAgent (GA)

We use the GA `azure-ai-projects` SDK (`>= 2.0`) to create a versioned **PromptAgent** in the Foundry project with the GA `BingGroundingTool` attached. The agent runs server-side inside Foundry; we invoke it from Python through `FoundryAgent` from `agent-framework-foundry`.


In [ ]:
# Imports for the GA Foundry control-plane SDK.
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    BingGroundingTool,
    BingGroundingSearchToolParameters,
    BingGroundingSearchConfiguration,
)


# ═════════════════════════════════════════════════════════════════════════════
# Transfer-Pricing Comparables Researcher — fixed system instructions
# Use case: batch generation of structured business descriptions for
#           comparable companies as part of a transfer-pricing analysis
#           (OECD TP Guidelines, FAR analysis terminology).
# ═════════════════════════════════════════════════════════════════════════════
TP_COMPARABLES_INSTRUCTIONS = """# Role

You are a transfer-pricing comparables researcher. You produce
structured, factual, audit-grade business descriptions of companies
from publicly available information, for use as comparable-company
documentation under the OECD Transfer Pricing Guidelines.

Tone: formal, analytical, sourced. Never editorialize, speculate, or
recommend. Present facts so the analyst (and ultimately a reviewing
tax authority) can draw their own comparability conclusions.

# Task

Given one target company per request, autonomously research it and
produce the report described by the user prompt's research objective.
Complete the full pipeline without asking follow-up questions. The
only exception is a genuinely ambiguous name (multiple distinct
entities matching the input) — state the ambiguity and proceed with
the entity that best matches the supplied attributes (ticker,
country, parent group).

# Rules

1. **Citations** — use only the grounding tool's inline markers
   (e.g. `【5:0†source】`) exactly as emitted; place markers
   back-to-back for multiple sources on one claim. Never write
   URLs, `[N]` footnotes, `[Source: URL]` tags, markdown links, or
   a `## Sources` / `## References` / bibliography section — the
   post-processor appends the authoritative source list from tool
   metadata and discards anything you write there. Every factual
   claim must carry at least one marker; if you can't ground it,
   drop the claim or write `Not found in public primary sources.`
   Cite as many sources as your searches actually yielded — sparse
   sourcing is a finding, not a failure.
2. **Source-quality hierarchy** — prefer in this order:
   (a) the company's own SEC filings (sec.gov/Archives/edgar/…),
       IR-site press releases, and official corporate website;
   (b) major newswires and reputable financial press (Reuters, FT,
       Bloomberg, WSJ);
   (c) industry-recognized databases (Crunchbase, PitchBook) — only
       when (a) and (b) don't cover the claim.
   **Never cite scraper / SEO aggregators** (stocklight.com,
   tradingview.com, simplywall.st, gurufocus.com, public.com,
   finbox.com, marketwatch quote pages, stockanalysis.com, yahoo
   finance summary pages, wisesheets.io, etc.). If an aggregator is
   your only option, drop the claim instead.
3. Honor the user prompt's section headings verbatim. If a required
   section has no public information, write
   `Not found in public sources.` — never omit it or fill it with
   speculation or analogies to similar companies.
4. If sources conflict on a fact (e.g., revenue), present both with
   their sources and note the discrepancy. Don't pick a winner.
5. If only consolidated group financials are public, say so — do not
   allocate group figures to the entity.
6. If the company has been acquired or wound down, state its current
   status and report the last known fiscal-year information.
7. Do not use paywalled or login-walled sources, or any non-public
   third-party data.
8. Do not opine on whether the company is a suitable comparable —
   describe the company; the analyst makes that call.

# Search strategy

Run multiple targeted searches per company to cover: legal entity
and ownership, business segments and products, geographic footprint
and end-markets, customer profile and distribution channels, key
competitors and market position, most-recent public financials
(revenue, headcount, segment splits), and functional-profile
signals (where R&D / manufacturing / IP sit, what risks the entity
appears to bear).

Bing's organic ranking tends to surface scrapers above primary
filings, so **run `site:`-restricted queries first**, before any
generic search:

- `<company name> 10-K site:sec.gov`           (U.S. issuers)
- `<company name> 10-Q site:sec.gov`           (latest quarterly)
- `<company name> annual report site:sec.gov` (proxies, 20-Fs, etc.)
- `<company name> site:<company-domain>.com`   (e.g. snowflake.com)
- `<company name> investor relations site:<company-domain>.com`
- For non-US issuers, substitute the relevant filing authority:
  `site:sec.gov.uk`, `site:sedarplus.ca`, `site:bundesanzeiger.de`,
  `site:asx.com.au`, etc.

Fall back to unrestricted search only if those return nothing
useful. If a fact still has no primary source after both, write
`Not found in public primary sources.`

# Functional Analysis (FAR)

When asked for a functional profile, address all three OECD
dimensions, asserting only what public sources support:

- **Functions performed** — e.g., R&D, manufacturing (full-fledged
  vs. contract / toll), marketing, distribution (full-risk vs.
  limited-risk / commissionaire), procurement, after-sales service,
  group treasury, IP management.
- **Assets used** — tangible (plant, distribution infrastructure)
  and intangible (trademarks, patents, know-how, customer
  relationships). Note where the entity legally owns IP vs. merely
  uses group IP.
- **Risks assumed** — market, inventory, credit, FX, product
  liability / warranty, R&D / technology. Identify which risks the
  entity bears vs. which sit with a related-party principal."""


def _build_bing_tool() -> BingGroundingTool:
    """Construct the BingGroundingTool spec bound to the project connection.

    Tuning knobs we set (and why):
      * count=20      — default is small (~5-10); a larger result set lets the
                        model pick from primary sources (SEC EDGAR, company IR
                        pages) instead of scraper sites that out-rank them on
                        a short result list (stocklight, tradingview, etc.).
      * market="en-US" — pin to the US/EN index. Without this, results drift
                        toward whatever locale the project happens to default
                        to, which gives inconsistent citations across rows.
      * set_lang="en" — restricts results to English-language pages.
    """
    return BingGroundingTool(
        bing_grounding=BingGroundingSearchToolParameters(
            search_configurations=[
                BingGroundingSearchConfiguration(
                    project_connection_id=BING_CONNECTION_ID,
                    count=20,
                    market="en-US",
                    set_lang="en",
                )
            ]
        )
    )


def ensure_agent() -> tuple[str, str]:
    """Create (or update) the transfer-pricing comparables PromptAgent. Returns (name, version)."""
    with (
        DefaultAzureCredential() as credential,
        AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential) as project_client,
    ):
        agent = project_client.agents.create_version(
            agent_name=AGENT_NAME,
            definition=PromptAgentDefinition(
                model=MODEL_DEPLOYMENT_NAME,
                instructions=TP_COMPARABLES_INSTRUCTIONS,
                tools=[_build_bing_tool()],
            ),
            description="Bing-grounded transfer-pricing comparables researcher: structured business descriptions with FAR analysis for OECD TP documentation.",
        )
    print(f"Agent ready: name={AGENT_NAME} version={getattr(agent, 'version', '?')}")
    return AGENT_NAME, str(getattr(agent, 'version', ''))


AGENT_NAME, AGENT_VERSION = ensure_agent()


## Single-run sanity check


In [ ]:
import asyncio
from azure.identity.aio import AzureCliCredential
import warnings
warnings.filterwarnings(
    "ignore",
    message=r".*experimental.*",
    category=UserWarning,
)
from agent_framework.foundry import FoundryAgent


def _format_with_citations(result) -> str:
    """Resolve grounding annotations into inline `[N](url)` markers and a
    deduped, authoritative `## Sources` section.

    In `agent-framework-foundry` the response shape is:
      AgentResponse.messages: list[Message]
        Message.contents: list[Content]   # TextContent etc.
          TextContent.text: str
          TextContent.annotations: Sequence[Annotation]

    `Annotation` is a TypedDict (dict-shaped) — *not* an object — with
    keys: `url`, `title`, `snippet`, and `annotated_regions` (a list of
    `TextSpanRegion` dicts each carrying `start_index`/`end_index`).
    There is no marker substring inside the text to find-and-replace,
    so we splice the citation in at each region's `end_index`.

    Strategy:
      1. For every TextContent, gather (end_index, url, title) tuples
         from each annotation's `annotated_regions`. Build a deduped
         URL→index map so the same source gets one footnote number.
      2. Splice `[N](url)` markers into the text right-to-left so
         insertions don't invalidate later offsets.
      3. Regex-scrub any orphan `【N:M†source】` / `[N:M+source]`
         markers the model might still emit if it ignored instructions.
      4. Strip any model-written `## Sources` / `## References` block
         (defensive: an earlier prompt asked for one, and models can
         regress to that habit producing hallucinated URLs).
      5. Append the authoritative `## Sources` list from the grounding
         metadata.
    """
    import re as _re

    _MARKER_RE = _re.compile(
        r"【\d+:\d+†[^】]*】"   # 【5:0†source】
        r"|\[\d+:\d+\+[^\]]*\]"  # [5:0+source]
    )
    _MODEL_SOURCES_RE = _re.compile(
        r"\n\s*#{1,6}\s*(?:Sources|References|Citations|Bibliography)\b.*\Z",
        _re.IGNORECASE | _re.DOTALL,
    )

    def _get(ann, key, default=None):
        # Annotation is a TypedDict in agent-framework-foundry, but be
        # tolerant of object-shaped annotations from other SDK builds.
        if isinstance(ann, dict):
            return ann.get(key, default)
        return getattr(ann, key, default)

    parts: list[str] = []
    sources: list[tuple[str, str]] = []
    url_to_idx: dict[str, int] = {}

    for msg in getattr(result, "messages", []) or []:
        for content in getattr(msg, "contents", []) or []:
            text = getattr(content, "text", None)
            if not isinstance(text, str) or not text.strip():
                continue

            splices: list[tuple[int, str]] = []
            for ann in getattr(content, "annotations", None) or []:
                url = _get(ann, "url") or _get(ann, "uri")
                if not url:
                    continue
                title = _get(ann, "title") or url
                if url not in url_to_idx:
                    url_to_idx[url] = len(sources) + 1
                    sources.append((url, title))
                idx = url_to_idx[url]

                regions = _get(ann, "annotated_regions") or []
                placed = False
                for region in regions:
                    end = _get(region, "end_index")
                    if isinstance(end, int) and 0 <= end <= len(text):
                        splices.append((end, f" [{idx}]({url})"))
                        placed = True
                # Fall back: some SDK builds put start_index/end_index
                # directly on the annotation, or expose a marker substring.
                if not placed:
                    marker = _get(ann, "text")
                    if isinstance(marker, str) and marker and marker in text:
                        text = text.replace(marker, f" [{idx}]({url})")
                        continue
                    end = _get(ann, "end_index")
                    if isinstance(end, int) and 0 <= end <= len(text):
                        splices.append((end, f" [{idx}]({url})"))

            # Right-to-left so earlier offsets stay valid.
            for end, insert in sorted(splices, key=lambda x: x[0], reverse=True):
                text = text[:end] + insert + text[end:]

            parts.append(text)

    answer = "".join(parts) if parts else (getattr(result, "text", "") or "")
    answer = _MARKER_RE.sub("", answer)
    answer = _MODEL_SOURCES_RE.sub("", answer).rstrip()

    if sources:
        answer += "\n\n## Sources\n"
        for i, (url, title) in enumerate(sources, 1):
            answer += f"{i}. [{title}]({url})\n"
    return answer


async def run_research(question: str) -> str:
    """Send a single question to the transfer-pricing comparables agent and
    return the answer with citation placeholders resolved to markdown links."""
    async with (
        AzureCliCredential() as credential,
        FoundryAgent(
            project_endpoint=PROJECT_ENDPOINT,
            agent_name=AGENT_NAME,
            agent_version=AGENT_VERSION,
            credential=credential,
        ) as agent,
    ):
        result = await agent.run(question)
        return _format_with_citations(result)


# Sanity-check the agent on a single TP-comparables prompt before we scale to the bulk runner.
answer = await run_research(
    "Produce a transfer-pricing comparables business description for Snowflake Inc. "
    "in the cloud data warehouse / lakehouse segment."
)
# Print the full answer
print(answer)


## Bulk company research — transfer-pricing comparables at scale

The bulk runner below scales the agent to hundreds-to-thousands of
comparable companies, with the throughput / SLA characteristics typical
of a transfer-pricing comparables batch (1–2 day turnarounds).

Two design properties matter:

- **One-shot per company.** Each row is an independent agent invocation.
  There is no step-level fan-out / fan-in; the "unit of parallelism" is
  the *whole agent run*.
- **Accuracy + consistency are critical.** The research objective is
  fixed across the batch, so output shape must be identical across rows
  for downstream comparable-company analysis.

What the runner provides:

| Concern | How this notebook handles it |
|---|---|
| Foundry-native execution | Each row runs against the GA `TransferPricingComparablesAgent` provisioned above. Server-side, in tenant, with Bing grounding. |
| In-process parallelism | `asyncio` + a semaphore caps concurrency. For very large jobs the same `run_research(...)` call is the work item you'd queue into **Azure Container Apps jobs** or **Service Bus / Storage Queue** workers. |
| Output consistency | A typed `CompanyResearchQuery` carries the fixed research objective + per-row entity attributes into a deterministic prompt template — every row uses the byte-identical instruction shape. |
| Agent-level (not step-level) parallelism | Each agent run is an independent task: `asyncio.gather` over `run_research(...)` calls. No intermediate steps to aggregate. |
| Partial-failure recovery | Per-row exceptions, configurable retries with exponential backoff, and a resumable JSONL checkpoint — re-running a job only re-issues the still-pending rows. |


In [ ]:
import asyncio
import json as _json
import time
from dataclasses import dataclass, field
from pathlib import Path


@dataclass
class CompanyResearchQuery:
    """One row in a bulk company-research batch.

    The research objective is fixed across the batch — only the entity
    attributes vary. Keeping them in separate fields prevents accidental
    prompt drift between rows and keeps the checkpoint JSONL diff-friendly.
    """
    company_name: str
    attributes: dict[str, str] = field(default_factory=dict)
    row_id: str = ""

    def __post_init__(self) -> None:
        if not self.row_id:
            self.row_id = self.company_name


# Fixed research objective for the transfer-pricing comparables use case.
# Keep this string stable across a single batch — change it and you change the
# semantic shape of every row in the output dataset.
TRANSFER_PRICING_OBJECTIVE = (
    "Produce a structured business description suitable for a transfer-pricing "
    "comparables analysis. Use Bing grounding for every claim, and leave the "
    "grounding tool's native citation markers in place at the end of each "
    "factual statement. Do not write URLs, footnote numbers, or a Sources "
    "section yourself — the post-processor appends the authoritative Sources "
    "list from the grounding annotations.\n\n"
    "Required sections (use these headings verbatim):\n"
    "  1. Company overview (legal entity, HQ, year founded, ownership status)\n"
    "  2. Business segments and products/services\n"
    "  3. Geographic footprint and material end-markets\n"
    "  4. Customer profile and primary distribution channels — enumerate\n"
    "     industry verticals served (e.g. Financial services; Advertising,\n"
    "     media & entertainment; Retail & consumer goods; Healthcare & life\n"
    "     sciences; Manufacturing; Technology; Telecom; Travel & hospitality;\n"
    "     Public sector). Use only verticals the company itself publicly\n"
    "     identifies; do not infer.\n"
    "  5. Key competitors and market position\n"
    "  6. Reported financials if publicly available (revenue, headcount, segment splits)\n"
    "  7. Functional profile (FAR analysis) — address EACH of the three\n"
    "     OECD dimensions as a separate sub-section with explicit bullets:\n"
    "\n"
    "     7.1 Functions performed — break down by function with one bullet\n"
    "         each, citing public evidence. Cover at minimum:\n"
    "           * R&D (where performed, scale, type)\n"
    "           * Manufacturing / production (full-fledged, contract, toll,\n"
    "             or N/A for software businesses)\n"
    "           * Product management and platform/cloud operations (for\n"
    "             SaaS/platform companies)\n"
    "           * Marketing and sales (direct sales, indirect channels,\n"
    "             partner ecosystem)\n"
    "           * Distribution (full-risk vs. limited-risk / commissionaire,\n"
    "             where applicable)\n"
    "           * Procurement\n"
    "           * Customer support and professional services\n"
    "           * Group treasury and corporate functions\n"
    "           * IP management and licensing\n"
    "\n"
    "     7.2 Assets used — separate bullets for tangible and intangible:\n"
    "           * Tangible: plant, equipment, distribution infrastructure,\n"
    "             owned data centers (note explicitly if the company relies\n"
    "             on third-party hyperscalers rather than owning its own)\n"
    "           * Intangible: trademarks and brand, patents, proprietary\n"
    "             software / platform IP, know-how and trade secrets,\n"
    "             customer relationships and ecosystem / network effects.\n"
    "             Note where the entity legally owns IP vs. merely uses\n"
    "             group IP when public sources support the inference.\n"
    "\n"
    "     7.3 Risks assumed — one bullet per risk category:\n"
    "           * Market and competitive risk\n"
    "           * Technology and R&D risk\n"
    "           * Service availability / operational risk (for cloud/SaaS)\n"
    "           * Data security, privacy and compliance risk\n"
    "           * Credit risk on customer receivables\n"
    "           * Foreign-exchange and international-operations risk\n"
    "           * Supply-chain or vendor-dependency risk (for software\n"
    "             companies relying on hyperscalers, name the dependency)\n"
    "           * Product liability / warranty risk (for product companies)\n"
    "         Identify, where supported by public sources, which risks the\n"
    "         entity bears vs. which sit with a related-party principal;\n"
    "         otherwise note explicitly that public information does not\n"
    "         support an entity-level risk allocation.\n\n"
    "Quality rules:\n"
    "  - If a section (or any 7.x sub-section bullet) has no public\n"
    "    information, write 'Not found in public sources' for that\n"
    "    specific bullet rather than omitting it.\n"
    "  - Do not speculate or interpolate from similar companies.\n"
    "  - Prefer primary sources (annual reports, regulatory filings, official websites)."
)


def render_company_prompt(query: CompanyResearchQuery, objective: str = TRANSFER_PRICING_OBJECTIVE) -> str:
    """Render one row into the deterministic prompt every batch member receives."""
    attrs = "\n".join(
        f"  - {k}: {v}" for k, v in sorted(query.attributes.items()) if v
    ) or "  (none provided)"
    return (
        f"Entity\n"
        f"------\n"
        f"Name: {query.company_name}\n"
        f"Attributes:\n{attrs}\n\n"
        f"Research objective\n"
        f"------------------\n"
        f"{objective}\n"
    )


async def _run_one_company(
    query: CompanyResearchQuery,
    *,
    objective: str,
    max_attempts: int,
    base_backoff_s: float,
) -> tuple[str, str | None, str | None]:
    """Run one company with bounded retries on transient failures."""
    prompt = render_company_prompt(query, objective=objective)
    last_err: Exception | None = None
    for attempt in range(1, max_attempts + 1):
        try:
            answer = await run_research(prompt)
            return query.row_id, answer, None
        except Exception as e:
            last_err = e
            if attempt < max_attempts:
                # Exponential backoff smooths 429 / transient 5xx retry storms.
                await asyncio.sleep(base_backoff_s * (2 ** (attempt - 1)))
    return query.row_id, None, f"{type(last_err).__name__}: {last_err}"


async def run_company_batch(
    queries: list[CompanyResearchQuery],
    *,
    objective: str = TRANSFER_PRICING_OBJECTIVE,
    concurrency: int = 16,
    max_attempts: int = 3,
    base_backoff_s: float = 2.0,
    checkpoint_path: str | Path | None = None,
    resume: bool = True,
) -> dict[str, dict]:
    """Run the agent across many companies in parallel.

    Features for the transfer-pricing scale point (hundreds/thousands of rows,
    1–2 day SLA):

    * **Agent-level parallelism.** Each row is a complete, independent agent
      run (`run_research`). The "unit of parallelism" is the whole agent run.
    * **In-process concurrency cap** via `asyncio.Semaphore(concurrency)`.
    * **Retries with exponential backoff** for transient failures.
    * **Resumable JSONL checkpoint.** Every completed row is appended
      immediately; re-running with `resume=True` skips rows already present.
    * **Scaling beyond one process.** The same `_run_one_company` coroutine
      is the work item for an Azure Container Apps job, a Service Bus or
      Storage Queue worker, or an Azure Functions queue-trigger. Agent
      identity (name + version) and `objective` travel with the message —
      nothing else changes.
    """
    completed: dict[str, dict] = {}

    ckpt = Path(checkpoint_path) if checkpoint_path else None
    if ckpt and resume and ckpt.exists():
        with ckpt.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rec = _json.loads(line)
                completed[rec["row_id"]] = rec
        print(f"Resumed from {ckpt}: {len(completed)} rows already complete.")

    pending = [q for q in queries if q.row_id not in completed]
    if not pending:
        print("All rows already complete — nothing to do.")
        return completed
    print(f"Running batch: {len(pending)} pending / {len(queries)} total "
          f"(concurrency={concurrency}, retries={max_attempts - 1}).")

    sem = asyncio.Semaphore(concurrency)
    ckpt_lock = asyncio.Lock()
    progress = {"done": 0, "ok": 0, "failed": 0}
    total = len(pending)
    started_at = time.monotonic()

    async def _worker(q: CompanyResearchQuery) -> None:
        async with sem:
            t0 = time.monotonic()
            row_id, answer, error = await _run_one_company(
                q,
                objective=objective,
                max_attempts=max_attempts,
                base_backoff_s=base_backoff_s,
            )
            rec = {
                "row_id": row_id,
                "company_name": q.company_name,
                "answer": answer,
                "error": error,
                "elapsed_s": round(time.monotonic() - t0, 2),
            }
            completed[row_id] = rec

            # Append under a lock so concurrent finishes don't interleave bytes.
            if ckpt:
                async with ckpt_lock:
                    with ckpt.open("a", encoding="utf-8") as f:
                        f.write(_json.dumps(rec) + "\n")

            progress["done"] += 1
            if error:
                progress["failed"] += 1
            else:
                progress["ok"] += 1
            if progress["done"] % 10 == 0 or progress["done"] == total:
                rate = progress["done"] / max(time.monotonic() - started_at, 1e-3)
                print(
                    f"  [{progress['done']:>5}/{total}] "
                    f"ok={progress['ok']} failed={progress['failed']} "
                    f"rate={rate:.1f}/s"
                )

    await asyncio.gather(*(_worker(q) for q in pending))
    elapsed = time.monotonic() - started_at
    print(
        f"Batch complete in {elapsed:.1f}s — "
        f"ok={progress['ok']} failed={progress['failed']} "
        f"(checkpoint: {ckpt if ckpt else 'disabled'})"
    )
    return completed


# ─────────────────────────────────────────────────────────────────────────────
# Demo: 5 comparable companies (replace with your real list of hundreds/thousands)
# In production this would be loaded from a CSV / database export.
# ─────────────────────────────────────────────────────────────────────────────
demo_companies = [
    CompanyResearchQuery(company_name="Snowflake Inc.",   attributes={"ticker": "SNOW", "country": "United States"}),
    CompanyResearchQuery(company_name="Databricks, Inc.", attributes={"country": "United States", "status": "private"}),
    CompanyResearchQuery(company_name="MongoDB, Inc.",    attributes={"ticker": "MDB",  "country": "United States"}),
    CompanyResearchQuery(company_name="Confluent, Inc.",  attributes={"ticker": "CFLT", "country": "United States"}),
    CompanyResearchQuery(company_name="Elastic N.V.",     attributes={"ticker": "ESTC", "country": "Netherlands"}),
]

# Small concurrency for the demo so notebook output stays readable.
# In production, start at 16 and tune based on observed throughput / 429 rate.
results = await run_company_batch(
    demo_companies,
    concurrency=3,
    max_attempts=2,
    checkpoint_path="_bulk_company_checkpoint.jsonl",
)

print("\n=== Completed rows ===")
for row_id, rec in results.items():
    print(f"\n--- {row_id} (elapsed {rec['elapsed_s']}s) ---")
    if rec["error"]:
        print(f"  FAILED: {rec['error']}")
    else:
        # Full answer (matches the single-run sanity check format:
        # inline `[N](url)` citations + the post-processor's `## Sources` block).
        print(rec["answer"] or "")

# Clean up demo checkpoint. In production, retain it for audit / resumability.
Path("_bulk_company_checkpoint.jsonl").unlink(missing_ok=True)


## Citation Guardrail Pipeline

### What Bing Grounding + DefaultV2 already solve

**Foundry with Bing Grounding** already addresses several common
citation-quality concerns at the platform level:

| # | Concern | Solved by | Notes |
|----|----------------------------------------------|-------------------------------------|-------------------------------------------------------|
| 1 | Broken or invalid links in citations | ⚠️ **Partially** — Bing + Layer 2 | Bing's index is fresher, but URLs still rot → Layer 2 |
| 2 | Prompt-engineering burden to validate cites | ✅ **Bing Grounding** | Structured annotations constrain the model to cite only search results |
| 3 | Unsafe/NSFW content at citation destinations | ✅ **Bing SafeSearch + DefaultV2** | SafeSearch=Strict filters adult results; DefaultV2 scans agent output for harmful text |
| 4 | Uncertainty around search indexing quality | ✅ **Bing Grounding** | Transparent, well-understood index with source attribution |
| 5 | Trustworthy citation sources from Bing | ✅ **Bing Grounding** | Exactly what they asked for |

### What still needs a custom guardrail

Only **Requirement 1 (broken links)** genuinely still needs runtime validation.
Bing reduces link rot significantly, but URLs can go stale between indexing and
response time. Layer 2 catches this.

Layers 1 and 3 are retained as **optional, defense-in-depth** checks that can
be enabled for high-stakes or compliance-sensitive use cases.

```
Agent answer (with Bing Grounding + DefaultV2 guardrail)
        │
        │  ┌─ Already handled by platform ────────────────────────┐
        │  │  • Bing SafeSearch=Strict filters unsafe sources     │
        │  │  • DefaultV2 scans output for harmful content        │
        │  │  • Grounding annotations constrain citations         │
        │  └──────────────────────────────────────────────────────┘
        ▼
  Layer 1 ─ Groundedness Detection (OPTIONAL — defense-in-depth)
        │   Adds value for: compliance-sensitive or regulated domains
        ▼
  Layer 2 ─ URL Reachability (REQUIRED)
        │   Catches link rot that Bing indexing alone cannot prevent
        ▼
  Layer 3 ─ Destination Content Safety (OPTIONAL — defense-in-depth)
        │   Adds value for: scenarios where SafeSearch filtering is insufficient
        ▼
  ✅ Return validated answer  ─or─  🔁 Ask agent to revise
```


### Layer 1 — Groundedness Detection (OPTIONAL — defense-in-depth)

**Original concern:** *Uncertainty around search indexing quality.*

**Now largely solved by:** Bing Grounding provides transparent source attribution —
the model is constrained to cite URLs from structured search results, not from its
own training data. This was the core issue with the raw OpenAI API approach.

**When to still enable this layer:**
- Regulated/compliance domains (legal, medical, financial) where you need a second
  opinion that claims are semantically grounded in source material
- When the model may paraphrase or extrapolate beyond what sources actually state

Set `enable_groundedness=True` in `run_research_guarded()` to activate.


In [ ]:
# Imports for Layer 1 (Groundedness Detection).
#   re                          : URL regex
#   httpx                       : async HTTP client (Content Safety REST call)
#   dataclass, field            : ergonomic result record
#   DefaultAzureCredential      : sync credential used to obtain a bearer token
import re
import httpx
from dataclasses import dataclass, field
from azure.identity import DefaultAzureCredential

# Permissive URL regex — matches anything starting with http(s):// up to a
# terminator that commonly appears next to a citation (whitespace, brackets,
# quotes). We strip trailing punctuation (".,;") in extract_urls() below.
URL_RE = re.compile(r"https?://[^\s)\]>\"']+")

# Content Safety Groundedness Detection is currently preview-only — pin the
# version explicitly so the call doesn't silently change behavior.
GROUNDEDNESS_API_VERSION = "2024-09-15-preview"


def extract_urls(text: str) -> list[str]:
    """Extract unique URLs from a string, preserving original order.

    Why dedupe? The agent often cites the same source multiple times; we don't
    want the guardrail to hit the same URL 5x.
    """
    seen: set[str] = set()
    out: list[str] = []
    for u in URL_RE.findall(text or ""):
        # Strip trailing sentence punctuation the regex captures by accident.
        u = u.rstrip(".,;")
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


def collect_grounding_text(run_result) -> list[str]:
    """Extract the *text snippets* the Bing tool returned during the run.

    These snippets become the `groundingSources` passed to the Groundedness
    Detection API — i.e. the "ground truth" that the agent's answer is
    semantically compared against.
    """
    sources: list[str] = []
    # An agent run is a thread of messages; each message has typed content parts.
    # We walk both levels defensively (getattr with [] fallback) because the
    # shape varies slightly between PromptAgent revisions.
    for msg in getattr(run_result, "messages", []) or []:
        for content in getattr(msg, "contents", []) or []:
            text_val = getattr(content, "text", None)
            if isinstance(text_val, str) and text_val.strip():
                sources.append(text_val.strip())
    return sources


def collect_grounded_urls(run_result) -> set[str]:
    """Pull every URL that Bing actually returned/annotated for this run.

    Used by Layer 2 to detect "hallucinated" citations — URLs the model
    mentions in the answer but that never appeared in Bing's tool output.
    """
    grounded: set[str] = set()
    for msg in getattr(run_result, "messages", []) or []:
        for content in getattr(msg, "contents", []) or []:
            # Direct URL attributes on content parts (some content types).
            for attr in ("url", "uri"):
                val = getattr(content, attr, None)
                if isinstance(val, str) and val.startswith("http"):
                    grounded.add(val)
            # Citation annotations attached to text content.
            ann = getattr(content, "annotations", None) or []
            for a in ann:
                u = getattr(a, "url", None) or getattr(a, "uri", None)
                if isinstance(u, str) and u.startswith("http"):
                    grounded.add(u)
            # Fallback: any URLs embedded in the raw tool text itself.
            text_val = getattr(content, "text", None)
            if isinstance(text_val, str):
                grounded.update(extract_urls(text_val))
    return grounded


# ── Layer 1: Groundedness Detection ──────────────────────────────────────────
# Requirement: "Uncertainty around search indexing quality"
# Requirement: "Need for prompt engineering to validate citations"
#
# Calls Content Safety detectGroundedness to semantically verify that the
# agent's claims are supported by the Bing search results. This replaces
# manual prompt-engineering tricks with an automated, API-driven check.


@dataclass
class GroundednessResult:
    """Aggregated Layer 1 verdict.

    Attributes:
        ungrounded_detected   : True if the API flagged any ungrounded claims.
        ungrounded_percentage : Fraction of the answer flagged (0.0–1.0).
        ungrounded_details    : Specific spans the API considered unsupported.
    """
    ungrounded_detected: bool
    ungrounded_percentage: float
    ungrounded_details: list[str] = field(default_factory=list)


async def check_groundedness(
    answer: str,
    grounding_sources: list[str],
    endpoint: str = CONTENT_SAFETY_ENDPOINT,
) -> GroundednessResult:
    """Layer 1: Verify the answer's claims are grounded in the Bing search results.

    Uses Content Safety `detectGroundedness` which performs *semantic* comparison
    — it catches fabricated/paraphrased claims that wouldn't show up in a naive
    URL-presence check.
    """
    # Mint an AAD token for the cognitiveservices audience (Content Safety).
    # DefaultAzureCredential will pick up `az login` creds or env-based creds.
    credential = DefaultAzureCredential()
    token = credential.get_token("https://cognitiveservices.azure.com/.default")

    # REST endpoint — Groundedness Detection isn't yet in the typed SDK client.
    url = f"{endpoint}/contentsafety/text:detectGroundedness?api-version={GROUNDEDNESS_API_VERSION}"

    # Request payload:
    #   domain  : "Generic" works for web research; use "Medical" for clinical text
    #   task    : "QnA" tells the model the input is question/answer (vs summarization)
    #   text    : the agent's answer, clipped to the API's per-call character cap
    #   groundingSources: chunks Bing surfaced; the API checks `text` against these
    #   reasoning: keep False to minimize latency; set True for human-readable rationale
    payload = {
        "domain": "Generic",
        "task": "QnA",
        "text": answer[:7500],  # API character limit
        "groundingSources": grounding_sources,
        "reasoning": False,
    }

    async with httpx.AsyncClient(timeout=30.0) as client:
        resp = await client.post(
            url,
            headers={"Authorization": f"Bearer {token.token}", "Content-Type": "application/json"},
            json=payload,
        )
        resp.raise_for_status()
        data = resp.json()

    # Unpack the API response into our dataclass, with safe defaults for
    # missing fields (the API has been evolving in preview).
    return GroundednessResult(
        ungrounded_detected=data.get("ungroundedDetected", False),
        ungrounded_percentage=data.get("ungroundedPercentage", 0.0),
        ungrounded_details=[d["text"] for d in data.get("ungroundedDetails", [])],
    )


### Layer 2 — URL Reachability (REQUIRED)

**Original concern:** *Broken or invalid links in citations.*

**Partially mitigated by:** Bing's index is significantly fresher than OpenAI's
opaque search, so fewer dead links will appear. However, URLs can go stale between
Bing's last crawl and the time the response is generated.

**Why this layer is still required:** Link rot is an inherent web problem that no
search engine fully prevents. A runtime HTTP check is the only way to guarantee
every cited link resolves at delivery time. This is the one requirement that
Bing + DefaultV2 cannot solve.


In [ ]:
# ── Layer 2: URL Reachability ────────────────────────────────────────────────
# Requirement: "Broken or invalid links in citations"
#
# HTTP HEAD (with GET fallback) on every cited URL. Also verifies the URL
# was present in the Bing grounding annotations — catches hallucinated links
# that look plausible but were never returned by the search tool.


@dataclass
class URLCheck:
    """Result of a single URL probe.

    `ok` returns True only when the URL is both *reachable* (2xx/3xx) and
    *grounded* (came from Bing's tool output). A reachable-but-ungrounded
    URL is treated as a citation failure because it indicates a hallucination.
    """
    url: str
    reachable: bool
    status: int | None
    grounded: bool  # URL appeared in Bing tool annotations
    error: str | None = None

    @property
    def ok(self) -> bool:
        return self.reachable and self.grounded


async def check_url_reachability(
    urls: list[str], grounded_urls: set[str]
) -> list[URLCheck]:
    """Layer 2: Verify every cited URL is reachable AND came from Bing results.

    Uses asyncio.gather to probe all URLs concurrently — typical answers cite
    3–10 sources, so serial probing would add multi-second latency for no reason.
    """
    # follow_redirects=True so a 301→200 chain is counted as reachable.
    # Spoof a friendly User-Agent — some sites 403 the default Python UA.
    async with httpx.AsyncClient(
        follow_redirects=True,
        timeout=10.0,
        headers={"User-Agent": "FoundryResearchAgent/1.0"},
    ) as client:

        async def _check(u: str) -> URLCheck:
            try:
                # Try HEAD first — saves bandwidth on big pages.
                r = await client.head(u)
                # A few hosts block HEAD outright (405) or rate-limit it (403).
                # Fall back to GET only for those status codes.
                if r.status_code in (405, 403):
                    r = await client.get(u)
                # "Grounded" matches exact URL or prefix overlap (handles trailing
                # slash / fragment differences between Bing's URL and the model's).
                is_grounded = (u in grounded_urls) or any(
                    u.startswith(g) or g.startswith(u) for g in grounded_urls
                )
                return URLCheck(
                    url=u,
                    reachable=200 <= r.status_code < 400,
                    status=r.status_code,
                    grounded=is_grounded,
                )
            except Exception as e:
                # Network failure / DNS / TLS = treat as unreachable.
                # We still record `grounded` so the report can distinguish
                # "real source that's temporarily down" from "made-up URL".
                return URLCheck(
                    url=u, reachable=False, status=None,
                    grounded=u in grounded_urls, error=str(e),
                )

        # Local alias so we don't shadow the outer-cell `asyncio` import.
        import asyncio as _a
        results = await _a.gather(*(_check(u) for u in urls))
    return list(results)


### Layer 3 — Destination Content Safety (OPTIONAL — defense-in-depth)

**Original concern:** *A citation link redirected to unsafe/NSFW content.*

**Now largely solved by:**
- **Bing SafeSearch=Strict** (default for Bing Grounding) — filters adult/unsafe
  search results before they ever reach the model
- **DefaultV2 guardrail** — scans the agent's output text for hate, sexual,
  self-harm, and violence content

**When to still enable this layer:**
- Client-facing deliverables in highly regulated industries
- When SafeSearch may not catch edge cases (e.g., a page that was safe when indexed
  but later had its content changed)

Set `enable_destination_safety=True` in `run_research_guarded()` to activate.


In [ ]:
# ── Layer 3: Destination Content Safety ──────────────────────────────────────
# Requirement: "Safety concerns with citation destinations"
#
# Fetches the first ~2000 chars of each destination page and runs
# Content Safety text:analyze to flag hate/sexual/self-harm/violence.
# This catches the NSFW-redirect scenario where a stale or hijacked URL
# now returns unsafe content even though the original source was legitimate.

from azure.ai.contentsafety import ContentSafetyClient
from azure.ai.contentsafety.models import AnalyzeTextOptions

# Severity scale: 0=safe, 2=low, 4=medium, 6=high (the API returns even integers).
# Threshold of 2 means "flag anything above safe" — tighten to 4 if you only
# care about medium-and-above, loosen to 0 for the strictest possible filter.
SAFETY_SEVERITY_THRESHOLD = 2


@dataclass
class DestinationSafetyCheck:
    """Result of a single destination-page safety scan.

    `categories` maps each flagged category (Hate/Sexual/SelfHarm/Violence)
    to its detected severity, so the caller can decide which categories
    actually matter for their use case.
    """
    url: str
    safe: bool
    categories: dict[str, int] = field(default_factory=dict)  # category -> severity
    error: str | None = None


async def check_destination_safety(
    urls: list[str],
    endpoint: str = CONTENT_SAFETY_ENDPOINT,
    max_chars: int = 2000,
) -> list[DestinationSafetyCheck]:
    """Layer 3: Fetch each URL and scan the page content for harmful material.

    Note: this is a *destination* check, not a *response* check — we're verifying
    where the citations send the user, not what the agent itself wrote.
    """
    # ContentSafetyClient is a sync client; we wrap each call in a regular
    # httpx fetch + sync analyze_text call. The whole function is `async`
    # to compose with the rest of the pipeline.
    credential = DefaultAzureCredential()
    cs_client = ContentSafetyClient(endpoint, credential)

    results: list[DestinationSafetyCheck] = []

    async with httpx.AsyncClient(
        follow_redirects=True, timeout=10.0,
        headers={"User-Agent": "FoundryResearchAgent/1.0"},
    ) as http:
        for u in urls:
            try:
                # Fetch the destination page body (HTML).
                resp = await http.get(u)
                # Rough HTML→text strip. Good enough for safety scanning —
                # we don't need clean prose, just the words on the page.
                import re as _re
                page_text = _re.sub(r"<[^>]+>", " ", resp.text)[:max_chars]

                # Empty page (e.g. JS-heavy SPA returns blank HTML to httpx):
                # we can't judge what isn't there — treat as safe.
                if not page_text.strip():
                    results.append(DestinationSafetyCheck(url=u, safe=True))
                    continue

                # Run Content Safety analysis on the extracted text.
                analysis = cs_client.analyze_text(
                    AnalyzeTextOptions(text=page_text)
                )
                # The API returns one entry per category — collect them all.
                cats = {
                    item.category: item.severity
                    for item in analysis.categories_analysis
                }
                # "Safe" = every category sits below the configured threshold.
                is_safe = all(sev < SAFETY_SEVERITY_THRESHOLD for sev in cats.values())
                results.append(DestinationSafetyCheck(url=u, safe=is_safe, categories=cats))
            except Exception as e:
                # Treat unreachable pages as unsafe out of caution — better to
                # drop a possibly-fine citation than ship a possibly-bad one.
                results.append(DestinationSafetyCheck(url=u, safe=False, error=str(e)))

    return results


### Combined Pipeline with Optional Layers

The pipeline always runs **Layer 2 (URL reachability)** — the only requirement not
covered by Bing Grounding + DefaultV2. Layers 1 and 3 are opt-in for high-stakes
use cases.

**Requirement addressed:** *Need for prompt engineering to validate citations* —
the automated pipeline replaces manual prompt tricks entirely, regardless of
which optional layers are enabled.


In [ ]:
# ── Combined pipeline ────────────────────────────────────────────────────────
#
# Layer 2 (URL reachability) always runs — it's the one check Bing Grounding
# + DefaultV2 cannot replace.
#
# Layers 1 & 3 are opt-in via flags. Enable them for regulated domains or when
# extra assurance is needed beyond platform-level protections.


@dataclass
class GuardrailReport:
    """Aggregated results from the guardrail pipeline."""
    groundedness: GroundednessResult | None
    url_checks: list[URLCheck]
    safety_checks: list[DestinationSafetyCheck] | None

    @property
    def passed(self) -> bool:
        if self.groundedness and self.groundedness.ungrounded_detected:
            return False
        if not all(c.ok for c in self.url_checks):
            return False
        if self.safety_checks and not all(c.safe for c in self.safety_checks):
            return False
        return True

    def summary(self) -> str:
        lines = ["╔══ Citation Guardrail Report ══╗"]
        if self.groundedness:
            g = self.groundedness
            status = "PASS" if not g.ungrounded_detected else "FAIL"
            lines.append(f"  Layer 1 (Groundedness):  [{status}]  ungrounded={g.ungrounded_percentage:.0%}")
            for detail in g.ungrounded_details:
                lines.append(f"    ⚠ {detail[:120]}")
        else:
            lines.append("  Layer 1 (Groundedness):  [SKIP]  covered by Bing Grounding")

        for c in self.url_checks:
            flag = "PASS" if c.ok else "FAIL"
            lines.append(
                f"  Layer 2 (Reachability): [{flag}] {c.url}  "
                f"status={c.status} grounded={c.grounded}"
                + (f" error={c.error}" if c.error else "")
            )

        if self.safety_checks:
            for s in self.safety_checks:
                flag = "PASS" if s.safe else "FAIL"
                lines.append(
                    f"  Layer 3 (Safety):       [{flag}] {s.url}  "
                    f"categories={s.categories}"
                    + (f" error={s.error}" if s.error else "")
                )
        else:
            lines.append("  Layer 3 (Safety):       [SKIP]  covered by Bing SafeSearch + DefaultV2")

        overall = "✅ ALL PASSED" if self.passed else "❌ ISSUES DETECTED"
        lines.append(f"╚══ {overall} ══╝")
        return "\n".join(lines)


async def run_guardrail_pipeline(
    result,
    enable_groundedness: bool = False,
    enable_destination_safety: bool = False,
) -> GuardrailReport:
    """Run the guardrail pipeline on a single agent result."""
    grounding_text = collect_grounding_text(result)
    grounded_urls = collect_grounded_urls(result)
    cited_urls = extract_urls(result.text)

    groundedness = None
    if enable_groundedness:
        groundedness = await check_groundedness(result.text, grounding_text)

    url_checks = await check_url_reachability(cited_urls, grounded_urls)

    safety_checks = None
    if enable_destination_safety:
        reachable_urls = [c.url for c in url_checks if c.reachable]
        safety_checks = await check_destination_safety(reachable_urls)

    return GuardrailReport(
        groundedness=groundedness,
        url_checks=url_checks,
        safety_checks=safety_checks,
    )


async def run_research_guarded(
    question: str,
    max_revisions: int = 1,
    enable_groundedness: bool = False,
    enable_destination_safety: bool = False,
) -> str:
    """Run the transfer-pricing comparables agent with automated citation validation + revision loop."""
    async with (
        AzureCliCredential() as credential,
        FoundryAgent(
            project_endpoint=PROJECT_ENDPOINT,
            agent_name=AGENT_NAME,
            agent_version=AGENT_VERSION,
            credential=credential,
        ) as agent,
    ):
        session = agent.create_session()
        result = await agent.run(question, session=session)

        for attempt in range(max_revisions + 1):
            report = await run_guardrail_pipeline(
                result,
                enable_groundedness=enable_groundedness,
                enable_destination_safety=enable_destination_safety,
            )
            print(report.summary())

            if report.passed:
                return result.text

            if attempt == max_revisions:
                raise RuntimeError(
                    "Citation guardrail failed after revision. "
                    f"Report:\n{report.summary()}"
                )

            issues: list[str] = []
            if report.groundedness and report.groundedness.ungrounded_detected:
                issues.append(
                    "Ungrounded claims (not supported by search results):\n"
                    + "\n".join(f"  - {d}" for d in report.groundedness.ungrounded_details)
                )

            bad_urls = [c for c in report.url_checks if not c.ok]
            if bad_urls:
                issues.append(
                    "Broken or ungrounded URLs:\n"
                    + "\n".join(
                        f"  - {c.url} (reachable={c.reachable}, grounded={c.grounded})"
                        for c in bad_urls
                    )
                )

            if report.safety_checks:
                unsafe_urls = [s for s in report.safety_checks if not s.safe]
                if unsafe_urls:
                    issues.append(
                        "URLs with unsafe destination content (must be removed):\n"
                        + "\n".join(f"  - {s.url}" for s in unsafe_urls)
                    )

            revise_prompt = (
                "Your previous answer failed automated citation validation:\n\n"
                + "\n\n".join(issues)
                + "\n\nRevise your answer: remove any broken or ungrounded "
                "citations. Only keep claims supported by valid, reachable URLs "
                "from the Bing search results. Search again if needed."
            )
            result = await agent.run(revise_prompt, session=session)

        return result.text


# ─────────────────────────────────────────────────────────────────────────────
# Demo: guarded TP-comparables business description (Layer 2 only — the cheap default).
# For client-facing deliverables, enable groundedness + destination safety.
# ─────────────────────────────────────────────────────────────────────────────
print("=== Guarded TP-comparables business description ===\n")
tp_answer = await run_research_guarded(
    "Produce a transfer-pricing comparables business description for Snowflake Inc., "
    "and BigQuery in the cloud data warehouse / lakehouse segment."
)
print("\n=== Validated report ===\n")
print(tp_answer[:1500] + ("..." if len(tp_answer) > 1500 else ""))


## Batch Groundedness Evaluation

While the runtime pipeline validates **each individual request**, this evaluator
measures groundedness **across a test dataset** for:

- **Comparing approaches** — quantify quality differences across prompt / model / grounding configurations
- **Baseline & regression tracking** — catch quality drops after prompt/model changes
- **Stakeholder reporting** — optionally upload results to the Foundry portal for dashboards

Uses `azure-ai-evaluation`'s built-in `GroundednessEvaluator` (LLM-as-judge,
semantic groundedness 1–5).

URL-reachability is deliberately not measured here: Bing grounding guarantees
every cited URL came from a real search result, so reachability is near-100%
by construction and adds no regression signal.

### Picking a judge model

`GroundednessEvaluator` sends `max_tokens` to the judge. **`gpt-5.x` deployments
reject `max_tokens`** (they require `max_completion_tokens`) and the request fails
with HTTP 400. Two ways to handle this:

1. **Recommended** — provision a separate judge deployment of a model that still
   accepts `max_tokens` (e.g. `gpt-4o`, `gpt-4.1`, `gpt-4o-mini`) in the same
   Foundry account, and set `EVAL_JUDGE_DEPLOYMENT_NAME=<that-deployment>` in
   your `.env`. The judge doesn't need to match the agent model.
2. **Skip the eval cell entirely** if no compatible judge is available — leave
   `EVAL_JUDGE_DEPLOYMENT_NAME` unset and don't run the evaluation cells.


In [ ]:
# Imports for the batch-evaluation harness.
#   GroundednessEvaluator : built-in semantic groundedness judge (LLM-as-judge)
#   evaluate              : test-set runner that fans out queries and aggregates scores
from azure.ai.evaluation import GroundednessEvaluator, evaluate


# ── Judge model selection ──────────────────────────────────────────
# `GroundednessEvaluator` sends `max_tokens` to the judge. gpt-5.x deployments
# reject `max_tokens` (they require `max_completion_tokens`), so the judge
# typically points at a separate gpt-4o / gpt-4.1 deployment.
#
# Set EVAL_JUDGE_DEPLOYMENT_NAME in .env to the name of a gpt-4o / gpt-4.1
# deployment in the same Foundry account.
EVAL_JUDGE_DEPLOYMENT_NAME = os.environ.get("EVAL_JUDGE_DEPLOYMENT_NAME", "").strip()

if not EVAL_JUDGE_DEPLOYMENT_NAME:
    raise RuntimeError(
        "EVAL_JUDGE_DEPLOYMENT_NAME is not set. Set it in .env to a gpt-4o / "
        "gpt-4.1 deployment in the same Foundry account (gpt-5.x deployments "
        "reject the max_tokens parameter the evaluator sends and will 400)."
    )

# Strip the `/api/projects/...` suffix to get the bare AI Services endpoint
# the evaluator's underlying OpenAI client expects.
eval_model_config = {
    "azure_endpoint": PROJECT_ENDPOINT.split("/api/")[0],
    "azure_deployment": EVAL_JUDGE_DEPLOYMENT_NAME,
}
groundedness_evaluator = GroundednessEvaluator(eval_model_config)
print(f"GroundednessEvaluator ready (judge deployment: {EVAL_JUDGE_DEPLOYMENT_NAME})")


### Run batch evaluation

Provide a JSONL file with test queries. Each row is scored by the built-in
groundedness evaluator, then averaged across the dataset.

To publish results to the Foundry portal for dashboarding, pass
`azure_ai_project=...` to `evaluate()` (commented out below — fill in your
subscription / resource group / project to enable upload).


In [ ]:
# Example: create a small test dataset inline (replace with your own JSONL file).
#
# The dataset has ONE company per row — the TP-comparables agent's input model
# is a single tested-party / candidate-comparable, not a pair.
import asyncio
import json as _json
from pathlib import Path

test_queries = [
    {"query": "Produce a transfer-pricing comparables business description for Snowflake Inc."},
    {"query": "Produce a transfer-pricing comparables business description for Datadog, Inc."},
    {"query": "Produce a transfer-pricing comparables business description for Stripe, Inc."},
]

test_data_path = Path("_eval_test_queries.jsonl")
with test_data_path.open("w", encoding="utf-8") as f:
    for q in test_queries:
        f.write(_json.dumps(q) + "\n")


# `evaluate()` calls the target SYNCHRONOUSLY, one row at a time, on a worker
# thread. Our agent invocation is async, so we wrap it in `asyncio.run(...)`
# inside a plain sync function. asyncio.run() is safe here because each call
# happens on a worker thread with no pre-existing event loop.
async def _run_one(query: str) -> dict:
    async with (
        AzureCliCredential() as credential,
        FoundryAgent(
            project_endpoint=PROJECT_ENDPOINT,
            agent_name=AGENT_NAME,
            agent_version=AGENT_VERSION,
            credential=credential,
        ) as agent,
    ):
        result = await agent.run(query)
        context = "\n\n".join(collect_grounding_text(result))
        return {"response": result.text, "context": context}


def research_target(query: str) -> dict:
    """Sync target for `evaluate()`. Returns {response, context} for each row."""
    return asyncio.run(_run_one(query))


evaluators = {"groundedness": groundedness_evaluator}
evaluator_config = {
    "groundedness": {
        "column_mapping": {
            "query":    "${data.query}",
            "response": "${target.response}",
            "context":  "${target.context}",
        },
    },
}


# Optional: upload results to the Foundry portal for dashboarding. Fill in your
# subscription / RG / project name and uncomment to enable.
# azure_ai_project = {
#     "subscription_id":  "<your-subscription-id>",
#     "resource_group_name": "<your-resource-group>",
#     "project_name":     "<your-foundry-project-name>",
# }

eval_results = evaluate(
    data=str(test_data_path),
    target=research_target,
    evaluators=evaluators,
    evaluator_config=evaluator_config,
    # azure_ai_project=azure_ai_project,  # uncomment to upload to Foundry portal
)

print("\n=== Evaluation Results (TransferPricingComparablesAgent) ===")
print(_json.dumps(eval_results.get("metrics", {}), indent=2))

# Per-row diagnostics — useful when a metric drops to see which company caused it.
print("\n=== Per-row scores ===")
for row in eval_results.get("rows", []):
    q = row.get("inputs.query", "")
    g = row.get("outputs.groundedness.groundedness")
    print(f"  groundedness={g}  | {q[:80]}")

test_data_path.unlink(missing_ok=True)
